In [6]:
import pandas as pd 
import numpy as _np 
import matplotlib.pyplot as plt 
import os 
import rasterio
from rasterio.transform import rowcol
from rasterio.transform import xy
from tqdm import tqdm
from rasterio.windows import Window

In [ ]:
df = pd.read_csv("dataset_points.csv")

In [7]:
inputs = _np.load("./inputs_stacked/inputs.npy", mmap_mode="r")
print(np.min(inputs[:, :, :, 6]))

0.0


In [ ]:
# windows Forming  for full one file (nefthou el file mera brk)
def window_forming(raster_path, df, height=7, width=7 , var=None):
    covariable_files = os.listdir(raster_path)
    covariable_num = len(covariable_files)

    datasets = [rasterio.open(os.path.join(raster_path, fname)) for fname in covariable_files]
    transforms = [ds.transform for ds in datasets]

    windows = []
    mid = height // 2

    for i in range(len(df)):
        lat = df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]
        window = np.empty((height, width, covariable_num))

        for idx, src in enumerate(datasets):
            transform = transforms[idx] 
            pixel_y, pixel_x = latlon_to_pixel(lat, lon, src, transform)

            try:
                win = Window(pixel_x - mid, pixel_y - mid, width, height)
                data = src.read(1, window=win)
                window[:, :, idx] = data
                print("l7a9t")

            except Exception as e:
                print("nan")
                window[:, :, idx] = np.nan  # Fill with NaNs on failure

        windows.append(window)

    for ds in datasets:
        ds.close()

    return windows

In [ ]:
def window_forming_tqdm(raster_path, df, height=7, width=7 , var=None):
    covariable_files = os.listdir(raster_path)
    covariable_num = len(covariable_files)

    datasets = [rasterio.open(os.path.join(raster_path, fname)) for fname in covariable_files]
    transforms = [ds.transform for ds in datasets]

    windows = []
    mid = height // 2

    for i in tqdm(range(len(df)), desc="Processing points"):
        lat = df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]
        window = np.empty((height, width, covariable_num))

        for idx, src in enumerate(datasets):
            transform = transforms[idx] 
            pixel_y, pixel_x = latlon_to_pixel(lat, lon, src, transform)
            try:
                win = Window(pixel_x - mid, pixel_y - mid, width, height)
                data = src.read(1, window=win)
                window[:, :, idx] = data

            except Exception as e:
                print("point i failed", idx)
                window[:, :, idx] = np.nan  # Fill with NaNs on failure

        windows.append(window)

    for ds in datasets:
        ds.close()

    return windows

In [ ]:
# for devided tiff files 
def window_forming2(raster_path  , df ,height=15 , width=15 ):
    covariable_num = 1
    windows = []

    for i in range(len(df)) :
        print(f"the dataset is in : {i}") 
        lat =  df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]
        window = np.empty((height, width, covariable_num))
        for index , j in enumerate(os.listdir(raster_path)):
            with rasterio.open(f"{raster_path}/{j}") as src :
                raster_data = src.read()
                transform = src.transform 
            try : 
                pixel_x ,pixel_y = latlon_to_pixel(lat , lon , src , transform)
                mid = height//2
                window[:,:,index] = raster_data[ 0 , pixel_x - mid : pixel_x + mid +1 , pixel_y - mid : pixel_y + mid +1 ]
                break 
            except Exception as e:
                print("can'found in ")
                continue 
        windows.append(window)
    return windows

In [ ]:
def window_forming3(all_files  , df ,height=15 , width=15 ):
    covariable_num = len(all_files)
    windows = []
    for i in range(len(df)) :
        print(f"the dataset is in : {i}") 
        lat =  df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]

        window = np.empty((height, width, covariable_num))

        for index , i in enumerate(all_files):
            raster_data = i.raster_data
            pixel_x ,pixel_y = latlon_to_pixel(lat , lon ,  i.src , i.transform)
            print(pixel_x , pixel_y)
            mid = height//2
            try : 
                window[:,:,index] = raster_data[ pixel_x - mid : pixel_x + mid +1 , pixel_y - mid : pixel_y + mid +1 ]
            except Exception as e:
                print("error is here ")
                print(e)
        windows.append(window)
    return windows

In [ ]:
# from joblib import Parallel, delayed
# import numpy as np
# import rasterio
# from rasterio.windows import Window
# from rasterio.transform import rowcol
# import os

# # Use your latlon_to_pixel function (the "wrong" one)
# def latlon_to_pixel(lat, lon, dataset, transform):
#     # simple bounds check
#     if not (dataset.bounds.left <= lon <= dataset.bounds.right and
#             dataset.bounds.bottom <= lat <= dataset.bounds.top):
#         lat -= 0.1
#         lon -= 0.1
#     row, col = rowcol(transform, lon, lat)
#     return row, col

# def extract_point_window_numpy(idx, lat, lon, raster_stack, dataset, transform, height=7, width=7):
#     mid_h = height // 2
#     mid_w = width // 2
#     num_covars, H, W = raster_stack.shape
#     window_stack = np.full((height, width, num_covars), np.nan, dtype=np.float32)

#     row, col = latlon_to_pixel(lat, lon, dataset, transform)

#     # Clip rows and columns to valid indices
#     row_start = max(row - mid_h, 0)
#     row_end   = min(row + mid_h + 1, H)
#     col_start = max(col - mid_w, 0)
#     col_end   = min(col + mid_w + 1, W)

#     window_slice = raster_stack[:, row_start:row_end, col_start:col_end].transpose(1, 2, 0)

#     # Place window_slice into center of output
#     start_r = mid_h - (row - row_start)
#     start_c = mid_w - (col - col_start)
#     window_stack[start_r:start_r+window_slice.shape[0], start_c:start_c+window_slice.shape[1], :] = window_slice

#     return window_stack


# def window_forming_parallel_numpy(raster_path, df, height=7, width=7, n_jobs=-1):
#     """Parallel window extraction using the old latlon_to_pixel function"""
#     cov_files = sorted(os.listdir(raster_path))
#     datasets = [rasterio.open(os.path.join(raster_path, f)) for f in cov_files]
    
#     # Load all rasters into memory
#     raster_stack = np.stack([ds.read(1) for ds in datasets], axis=0)
#     transform = datasets[0].transform
#     for ds in datasets:
#         ds.close()

#     # Parallel extraction
#     results = Parallel(n_jobs=n_jobs, backend="threading")(
#         delayed(extract_point_window_numpy)(
#             i, df.iloc[i]["X"], df.iloc[i]["Y"], raster_stack, datasets[0], transform, height, width
#         )
#         for i in tqdm(range(len(df)))
#     )

#     return results

# def window_forming_parallel_numpy_file(raster_path, df, height=7, width=7, n_jobs=-1):
#     """Parallel window extraction using the old latlon_to_pixel function"""
#     datasets = [rasterio.open(raster_path)]
    
#     # Load all rasters into memory
#     raster_stack = np.stack([ds.read(1) for ds in datasets], axis=0)
#     transform = datasets[0].transform
#     for ds in datasets:
#         ds.close()

#     # Parallel extraction
#     results = Parallel(n_jobs=n_jobs, backend="threading")(
#         delayed(extract_point_window_numpy)(
#             i, df.iloc[i]["X"], df.iloc[i]["Y"], raster_stack, datasets[0], transform, height, width
#         )
#         for i in tqdm(range(len(df)))
#     )

#     return results

In [ ]:
from joblib import Parallel, delayed
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.transform import rowcol
import os

from tqdm import tqdm

def extract_point_window_numpy(idx, lon, lat, raster_stack, dataset, transform, height=7, width=7):
    try:
        mid_h = height // 2
        mid_w = width // 2
        num_covars, H, W = raster_stack.shape

        # Convert lon/lat to raster row/col
        row, col = rasterio.transform.rowcol(transform, lon, lat)

        # Window boundaries
        r0 = row - mid_h
        r1 = row + mid_h + 1
        c0 = col - mid_w
        c1 = col + mid_w + 1

        # If window is outside the raster → return NaN window safely
        if (r0 < 0 or r1 > H or c0 < 0 or c1 > W):
            return np.full((height, width, num_covars), np.nan, dtype=np.float32)

        # Extract the window
        window_slice = raster_stack[:, r0:r1, c0:c1]

        # Validate shape
        if window_slice.shape != (num_covars, height, width):
            return np.full((height, width, num_covars), np.nan, dtype=np.float32)

        # Reorder to HWC
        return np.transpose(window_slice, (1, 2, 0))

    except Exception:
        # ANY unexpected runtime error → return safe NaN patch
        print("moushkil")
        return np.full((height, width, raster_stack.shape[0]), np.nan, dtype=np.float32)


def window_forming_parallel_numpy(raster_path, df, height=7, width=7, n_jobs=-1):
    """Parallel window extraction using the old latlon_to_pixel function"""
    cov_files = sorted(os.listdir(raster_path))
    datasets = [rasterio.open(os.path.join(raster_path, f)) for f in cov_files]
    
    # Load all rasters into memory
    raster_stack = np.stack([ds.read(1) for ds in datasets], axis=0)
    transform = datasets[0].transform
    for ds in datasets:
        ds.close()

    # Parallel extraction
    results = Parallel(n_jobs=n_jobs, backend="threading")(
        delayed(extract_point_window_numpy)(
            i, df.iloc[i]["X"], df.iloc[i]["Y"], raster_stack, datasets[0], transform, height, width
        )
        for i in tqdm(range(len(df)))
    )

    return results

def window_forming_parallel_numpy_file(raster_path, df, height=7, width=7, n_jobs=-1):
    """Parallel window extraction using the old latlon_to_pixel function"""
    datasets = [rasterio.open(raster_path)]
    
    # Load all rasters into memory
    raster_stack = datasets[0].read()
    transform = datasets[0].transform
    for ds in datasets:
        ds.close()

    # Parallel extraction
    results = Parallel(n_jobs=n_jobs, backend="threading")(
        delayed(extract_point_window_numpy)(
            i, df.iloc[i]["X"], df.iloc[i]["Y"], raster_stack, datasets[0], transform, height, width
        )
        for i in tqdm(range(len(df)))
    )

    return results

In [ ]:
def replace_nan_with_local_mean(data):
    data_filled = data.copy()
    B, H, W, C = data.shape

    for idx in tqdm(range(B), desc="Filling NaNs in batches"):
        for ch in range(C):
            matrix = data_filled[idx, :, :, ch]

            # Compute mean of all valid pixels in the 7x7 window
            valid_mean = np.nanmean(matrix)

            # Fallback if all are NaN
            if np.isnan(valid_mean):
                valid_mean = 0.0

            # Replace NaNs
            matrix = np.where(np.isnan(matrix), valid_mean, matrix)

            data_filled[idx, :, :, ch] = matrix

    return data_filled

In [ ]:
path = "./data/climat/"
dirs = [os.path.join(path, d) for d in os.listdir(path)]
for dir in dirs:
    name = os.path.splitext(os.path.basename(dir))[0]
    name = name.replace("wc2.1_", "")
    print("Starting with ->", name)
    climate = window_forming_parallel_numpy_file(dir , df)
    climate = np.array(climate)
    print("Finished with ->", name)
    missing_values = np.isnan(climate).sum()
    if missing_values > 1:
        print("missing values found: ", missing_values)
        climate = replace_nan_with_local_mean(climate)
        missing_values_removed = np.isnan(climate).sum()
        print("missing values now: ", missing_values_removed)
    np.save(f"./inputs/climat/climate_{name}.npy", climate)

In [ ]:
path = "./data/organisme/"
dirs = [os.path.join(path, d) for d in os.listdir(path)]
for dir in dirs:
    name = os.path.splitext(os.path.basename(dir))[0]
    print("Starting with ->", name)
    organisme = window_forming_parallel_numpy_file(dir , df)
    organisme = np.array(organisme)
    print("Finished with ->", name)
    missing_values = np.isnan(organisme).sum()
    if missing_values > 1:
        print("missing values found: ", missing_values)
        organisme = replace_nan_with_local_mean(organisme)
        missing_values_removed = np.isnan(organisme).sum()
        print("missing values now: ", missing_values_removed)
    np.save(f"./inputs/organisme/organisme_{name}.npy", organisme)

In [ ]:
path = "./data/relief/"
dirs = [os.path.join(path, d) for d in os.listdir(path)]
for dir in dirs:
    name = os.path.splitext(os.path.basename(dir))[0]
    print("Starting with ->", name)
    relief = window_forming_parallel_numpy_file(dir , df)
    relief = np.array(relief)
    print("Finished with ->", name)
    missing_values = np.isnan(relief).sum()
    if missing_values > 1:
        print("missing values found: ", missing_values)
        relief = replace_nan_with_local_mean(relief)
        missing_values_removed = np.isnan(relief).sum()
        print("missing values now: ", missing_values_removed)
    np.save(f"./inputs/relief/relief_{name}.npy", relief)

In [ ]:
dir = './data/ParentMaterial/rocksType_1km.tif'
name = os.path.splitext(os.path.basename(dir))[0]
print("Starting with ->", name)
parent_material = window_forming_parallel_numpy_file(dir , df)
parent_material = np.array(parent_material)
print("Finished with ->", name)
missing_values = np.isnan(parent_material).sum()
if missing_values > 1:
    print("missing values found: ", missing_values)
    parent_material = replace_nan_with_local_mean(parent_material)
    missing_values_removed = np.isnan(parent_material).sum()
    print("missing values now: ", missing_values_removed)
np.save(f"./inputs/parentMaterial/parentMaterial_{name}.npy", parent_material)

In [ ]:
sentinel = window_forming_parallel_numpy_file("./data/sentinel/Sentinel2_1km_WholeRegion.tif" , df)
sentinel = np.array(sentinel)
print(sentinel.shape)
# np.save("./inputs/sentinel/sentinel.npy", sentinel)

In [ ]:
np.isnan(sentinel).sum()

In [ ]:
def replace_nan_with_local_mean(data):
    data_filled = data.copy()
    B, H, W, C = data.shape

    for idx in tqdm(range(B), desc="Filling NaNs in batches"):
        for ch in range(C):
            matrix = data_filled[idx, :, :, ch]

            # Compute mean of all valid pixels in the 7x7 window
            valid_mean = np.nanmean(matrix)

            # Fallback if all are NaN
            if np.isnan(valid_mean):
                valid_mean = 0.0

            # Replace NaNs
            matrix = np.where(np.isnan(matrix), valid_mean, matrix)

            data_filled[idx, :, :, ch] = matrix

    return data_filled

sentinel = replace_nan_with_local_mean(sentinel)
print(np.isnan(sentinel).sum())

In [ ]:
np.save("./inputs/sentinel/sentinel.npy", sentinel)

In [ ]:
landuse = window_forming_parallel_numpy_file("./data/output1km/output_1km.tif" , df)
landuse = np.array(landuse)
print(landuse.shape)

In [ ]:
np.isnan(landuse).sum()

In [ ]:
def replace_nan_with_local_mean(data):
    data_filled = data.copy()
    B, H, W, C = data.shape

    for idx in tqdm(range(B), desc="Filling NaNs in batches"):
        for ch in range(C):
            matrix = data_filled[idx, :, :, ch]

            # Compute mean of all valid pixels in the 7x7 window
            valid_mean = np.nanmean(matrix)

            # Fallback if all are NaN
            if np.isnan(valid_mean):
                valid_mean = 0.0

            # Replace NaNs
            matrix = np.where(np.isnan(matrix), valid_mean, matrix)

            data_filled[idx, :, :, ch] = matrix

    return data_filled

landuse = replace_nan_with_local_mean(landuse)
print(np.isnan(landuse).sum())

In [ ]:
np.save("./inputs/landuse/landuse.npy", landuse)
np.save("./inputs_stacked/landuse.npy", landuse)

### 1.Climat

In [ ]:
climat_window = window_forming_parallel_numpy("./data/climat/" , df)
climat_window = np.array(climat_window)
np.save("./inputs/climat.npy", climat_window)

In [ ]:
climat_window.shape

In [ ]:
# Show overall information
print("Shape:", climat_window.shape)
print("Data type:", climat_window.dtype)

# Extract the first window
first_window = climat_window[0]

print("\nFirst window shape:", first_window.shape)
print("First window values:\n", first_window)

# Additional inspection
print("\nStats:")
print("Min:", np.nanmin(first_window))
print("Max:", np.nanmax(first_window))
print("NaNs:", np.isnan(first_window).sum())

### 2. organisme

In [ ]:
files = [f for f in os.listdir("./data/organisme/")]
files = sorted(files)
organisme = np.array([])
for file in files:
    print("processing ->", file)
    organisme = window_forming_parallel_numpy_file(os.path.join("./data/organisme/", file) , df)
    organisme = np.array(organisme)
    np.save(f"./inputs/organisme_{file.split('.')[0]}.npy", organisme)

### 3. relief

In [ ]:
files = [f for f in os.listdir("./data/relief/")]
files = sorted(files)
relief = np.array([])
for file in files:
    relief = window_forming_parallel_numpy_file(os.path.join("./data/relief/", file) , df)
    np.save(f"./inputs/relief_{file.split('.')[0]}.npy", relief)

### 4. Spatial Position

In [ ]:
def replace_nan_with_column_mean(data):
    """
    Fill NaN values in all channels of a 4D array (samples, height, width, channels)
    using the mean of vertical neighbors. If no valid neighbors, fill with 0.
    """
    data_filled = data.copy()
    num_samples, height, width, num_channels = data.shape

    for idx in range(num_samples):
        for ch in range(num_channels):
            matrix = data_filled[idx, :, :, ch]

            nan_indices = np.argwhere(np.isnan(matrix))
            for i, j in nan_indices:
                neighbors = []
                if i > 0:
                    neighbors.append(matrix[i-1, j])
                if i < height - 1:
                    neighbors.append(matrix[i+1, j])
                
                neighbors = [val for val in neighbors if not np.isnan(val)]
                
                if neighbors:
                    matrix[i, j] = np.mean(neighbors)
                else:
                    matrix[i, j] = 0

            data_filled[idx, :, :, ch] = matrix

    return data_filled

organisme

In [ ]:
def print_nan_counts_per_channel(data):
    """
    Print the number of NaN values for each channel in a 4D array:
    shape = (samples, height, width, channels)
    """
    num_channels = data.shape[3]
    for ch in range(num_channels):
        nan_count = np.isnan(data[:, :, :, ch]).sum()
        print(f"Channel {ch}: {nan_count} NaNs")

# Example usage:
print_nan_counts_per_channel(organisme)

In [ ]:
np.save("../Data_multi_CNN/landuse.npy" , landuse_windo)

### 4. Parent Material

In [ ]:
parent_window = window_forming_tqdm("./data/ParentMaterial/", df)
np.save("./data/Parent_Material.npy", parent_window)
np.array(parent_window).shape

In [ ]:
parent_window = window_forming_tqdm("./data/output1km/", df)
np.save("./data/ouput1km.npy", parent_window)
np.array(parent_window).shape